![alt text](1_Q6yA_1B_vsdGWAAwB8Z7rA.webp "Title")

![alt text](b6337e9122efe9c56985302d3123951ddd237ee6-840x331.webp "Title")

mükemmel—“**4a’ya kadar**” (yani veri hazırlığı + generator’lar + model kurulumu + `fit`’e gelene dek) olan kısmı **uzman perspektifinden** ama sade bir dille, tam not olacak şekilde anlatıyorum. İleride yeni model kurarken doğrudan bu notları referans alabilirsin.

---

# 1) CNN nedir? (kısa ama sağlam zihin modeli)

* **Amaç:** Görsellerdeki desenleri (kenar, doku, parça, şekil) **otomatik çıkarmak** ve sınıflandırmak.
* **Conv2D (evrişim):** Küçük bir çekirdek (örn. 3×3), görüntü üzerinde kayar; **aynı ağırlıklar paylaşılır** (weight sharing). Böylece hem parametre sayısı azalır hem de **konumdan bağımsız** özellik algılanır.
* **ReLU:** Doğrusal olmayanlık ekler; negatifleri 0’a kırpar → **hızlı öğrenme** ve **ölmeyen gradyan**.
* **Pooling (MaxPool):** Özellik haritasını altörnekler (2×2 gibi). **Hafif kaymalara dayanıklılık** ve parametre/hesap maliyeti düşer.
* **Flatten / GlobalAveragePooling (GAP):** Konvolüsyonlardan çıkan özellikleri tek vektöre çevirir (Flatten) veya her kanalın ortalamasını alır (GAP). Sonra **Dense** katman(lar) ile karar verilir.
* **Binary çıkış:** İkili sınıflama için `Dense(1, activation='sigmoid')` + `binary_crossentropy`.

---

# 2) Veri dizini ve generator’lar (ImageDataGenerator)

## 2.1 Klasör yapısı

Biz PetImages’ı **train/val/test** olarak ayırdık:

```
data/
  train/
    cats/
    dogs/
  val/
    cats/
    dogs/
  test/
    cats/
    dogs/
```

> `flow_from_directory` bu yapıyı **otomatik etiketler**: alt klasör adlarına göre `class_indices = {'cats': 0, 'dogs': 1}` gibi.

## 2.2 Neden `ImageDataGenerator`?

* **Belleğe yüklemez.** Tüm görselleri RAM’e almayız. Eğitim sırasında **batch batch** diskten okur.
* **Anında (on-the-fly) augment eder.** Her epoch’ta **farklı** (ama etiketi bozmayan) dönüşümler: döndürme, kaydırma, zoom, flip… → **genelleme artar, overfitting azalır**.
* **Ölçekleme (rescale)**: Pikseli `[0,255]` → `[0,1]` çeker (float32). Eğitim daha **stabil** olur.

## 2.3 Neden 160×160?

* CNN’ler sabit giriş boyutu ister. Farklı boyutlu resimleri **tek boyuta** getiririz.
* **160×160** hız–performans **dengesi**:

  * 128×128 → daha hızlı, biraz daha düşük doğruluk
  * 224×224 → biraz daha iyi doğruluk ama **daha yavaş ve bellek pahalı**
* Kediler/köpekler gibi basit veri için 128–160 aralığı çok uygundur.

## 2.4 Augmentation parametreleri (işlev ve doz)

* `rotation_range=15`: ±15° döndür → açıya dayanıklılık
* `width/height_shift_range=0.10`: %10 kaydır → kadraj değişimine dayanıklılık
* `zoom_range=0.10`: ±%10 zoom → farklı mesafelere dayanıklılık
* `shear_range=0.10`: hafif eğme → perspektif çeşitliliği
* `horizontal_flip=True`: ayna → kediler/köpekler için doğal
* **Kural:** “**Hafif ama çeşitli**”. Aşırı verirsen etiket gürültüsü yaratır (performans düşer).

## 2.5 Neden val/test’te augmentation yok?

* Değerlendirme **gerçeği** yansıtmalı. Validation/test’te yalnızca `rescale` yapılır. Augment edersen ölçüm **yanlı** olur.

## 2.6 Kod (ne oluyor, bellek nasıl kullanılıyor?)

```python
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE   = (160, 160)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.10, height_shift_range=0.10,
    shear_range=0.10, zoom_range=0.10,
    horizontal_flip=True
)
val_datagen  = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    "data/train", target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=True, seed=42
)
val_gen = val_datagen.flow_from_directory(
    "data/val", target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
)
test_gen = test_datagen.flow_from_directory(
    "data/test", target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
)
```

**Arka planda ne olur?**

* Her **batch**’te: dosyalar diskten okunur → `PIL` ile **resize (160×160)** → **augment** (sadece train) → **numpy float32** → **/255** ölçek → GPU’ya gönderilir.
* **Hepsi RAM’e yüklenmez.** Sadece o anda eğitilecek batch bellek kullanır. (Yaklaşık giriş bellek: 32×160×160×3×4 byte ≈ **\~9.8 MB**, aktivasyonlar hariç.)
* `shuffle=True` (train) karıştırır; `shuffle=False` (val/test) tekrarlanabilir ölçüm için.
* **Adım sayısı (steps\_per\_epoch)** = `len(train_gen)` = `ceil(num_train / batch_size)`. Keras bunu otomatik hesaplar.

---

# 3) Model kurulumunun mantığı (Conv → ReLU → Pool → Dense)

Örnek mimari (seninle kullandığımız):

```python
from tensorflow.keras import layers, models
from tensorflow.keras.metrics import AUC

def make_cnn(input_shape=(160,160,3), dropout=0.5):
    m = models.Sequential([
        layers.Conv2D(32, 3, activation='relu', padding='same', input_shape=input_shape),
        layers.MaxPool2D(2),                                  # 160→80
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPool2D(2),                                  # 80→40
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.MaxPool2D(2),                                  # 40→20
        layers.Flatten(),                                     # 20*20*128=51,200
        layers.Dropout(dropout),
        layers.Dense(128, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', AUC(name='auc')])
    return m
```

### Neden `input_shape=(160,160,3)`?

* Giriş resmin **boyutunu** ve **kanalını (RGB=3)** model bu katmanda görür. Sonraki katmanlar şekli otomatik devralır.

### `padding='same'` ne işe yarar?

* 3×3 konvolüsyon kenarlarda taşmasın diye **sıfır dolgusu** ekler, **genişlik/yükseklik korunur** (stride=1). Şekil takibi kolay olur: Conv sonrası aynı boyut, Pool sonrası yarı.

### Neden filtrer sayısı artar? 32→64→128

* Aşağı indikçe **soyut özellikler** artar. Pool ile uzamsal boyut azalırken kanal sayısını artırmak **temsil gücü** sağlar.

### Neden 3×3 kernel?

* Küçük, hesap ucuz. Üst üste birkaç 3×3, büyük bir **etkin alan** (receptive field) oluşturur.

### Flatten vs GAP (önemli!)

* Flatten + Dense(128) burada **\~6.5M parametre** üretir (asıl yük burada).
* **GlobalAveragePooling2D** kullanırsan parametre **çok azalır**, eğitim hızlanır, genelleme çoğu zaman gelişir:

  ```python
  layers.GlobalAveragePooling2D(),
  layers.Dropout(0.3),
  layers.Dense(128, activation='relu')
  ```

  > Geliştirmek istediğinde ilk deneyebileceğin iyileştirme budur.

---

# 4) Derleme (compile) ve kayıplar/metrikler

* **Loss:** `binary_crossentropy`

  $$
  \text{BCE} = -\frac{1}{N}\sum \big[y\log(\hat y) + (1-y)\log(1-\hat y)\big]
  $$

  Sigmoid çıkışla **doğal** eşleşir.
* **Optimizer:** `adam` (öğrenme oranını uyarlayan, hızlı ve kararlı).
* **Metrikler:** `accuracy` + **`AUC`** (ayrıştırma gücü; dengesizlikte daha anlamlı).

---

# 4a) Eğitimi başlatma (fit) — içeride neler olur?

```python
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early = EarlyStopping(monitor='val_auc', mode='max', patience=5,
                      restore_best_weights=True, verbose=1)
ckpt  = ModelCheckpoint("best_cnn.keras", monitor='val_auc', mode='max',
                        save_best_only=True, verbose=1)

history = model.fit(
    train_gen,                   # batch batch veri akacak (augment + rescale)
    validation_data=val_gen,     # augment yok, sadece rescale
    epochs=30,                   # üst sınır; EarlyStopping erken durdurur
    callbacks=[early, ckpt],
    verbose=1,
    # workers=4, use_multiprocessing=True,  # CPU'da IO/augment’i paralellemek için
)
```

**Eğitim döngüsünde ne olur?**

1. **Batch hazırlanır:** Generator diskten **N** görsel okur, `IMG_SIZE`’a resize eder, **train’de augment** uygular, `/255` ile ölçekler, `float32`’ye çevirir.
2. **İleri yayılım (forward):** Conv→ReLU→Pool…→Dense boyunca **aktivasyonlar** hesaplanır.
3. **Kayıp (loss):** BCE; gerçek etiketlerle kıyaslanır.
4. **Geri yayılım (backward):** Gradyanlar hesaplanır; **Adam** ağırlıkları günceller.
5. **Metric:** Accuracy/AUC güncellenir.
6. **Validation:** Her epoch sonunda **val\_gen** ile augment’siz değerlendirme.
7. **EarlyStopping:** `val_auc` **5 epoch** iyileşmezse eğitimi durdurur ve **en iyi ağırlıkları** geri yükler.
8. **ModelCheckpoint:** En iyi `val_auc` gördüğünde `best_cnn.keras` olarak kaydeder.

**Bellek ve hız notları**

* Giriş batch’i \~10 MB civarı; büyük bellek tüketen kısım **aktivasyonlar** ve **Flatten sonrası Dense**’tir.
* GPU varsa eğitim **çok hızlanır**. GPU yoksa `IMG_SIZE`→128, `BATCH_SIZE`→16 yapmak süreyi azaltır.
* **Parametre patlaması** (Flatten→Dense) nedeniyle eğitim yavaş geliyorsa **GAP** geçişi önerilir.

---

# 4a’ya kadar “en kritik” iyi uygulamalar (checklist)

* [x] **Train/Val/Test** ayrımı (val: erken durdurma, test: final skor)
* [x] **Bozuk görseller** elendi (PetImages’ta önemli)
* [x] **Augment sadece train’de**; val/test’te **yok**
* [x] **Rescale her yerde** (0–1)
* [x] **input\_shape** ilk Conv veya `layers.Input`’ta
* [x] **padding='same'** ile boyut yönetimi kolay
* [x] **Adam + BCE + Accuracy + AUC** başlangıç için doğru seçim
* [x] **EarlyStopping + Checkpoint** (pratikte şart gibi)
* [x] (Performans gerekirse) **Flatten → GAP** geçişi planlı

---

## Sık sorulanlar

* **“Tüm resimler RAM’e mi yükleniyor?”**
  Hayır. Sadece **o anki batch** RAM’dedir. Generator diskten okur, augment eder, modele verir. Sonra sonraki batch.

* **“Augment neden sadece train’de?”**
  Çünkü ölçümü bozmak istemeyiz. Değerlendirme **gerçek dağılım** üzerinde yapılmalı.

* **“Neden padding='same'?”**
  Boyut korunsun, kenar bilgisi kaybolmasın; şekil takibi kolaylaşsın.

* **“Niye sigmoid + BCE?”**
  İkili sınıflama; sigmoid (0–1 olasılık) ile BCE matematiksel olarak doğru ikili log-loss.

* **“Neden AUC?”**
  Eşik bağımsız **ayrıştırma** ölçer; sınıf dengesizliğinde accuracy’den daha anlamlı olabilir.

---

Hazırsan, **4b** (öğrenme eğrilerini çizip yorumlama) ve **5** (test + confusion matrix) adımlarına geçebiliriz. İstersen GAP’lı mimariyi de bir kopya olarak kurup aynı eğitimle karşılaştıralım; çoğu zaman **daha hızlı** ve **daha iyi genelleme** veriyor.


muhteşem—CNN tekrarını “defter notu” kıvamında, **baştan sona** hem kuramı hem pratiğiyle tek yerde topladım. Aşağıda:

* CNN nedir, neden işe yarar?
* Veriyi (PetImages) nasıl hazırlarız, neden böyle yaparız?
* Augmentation ayarlarının anlamı ve doğru doz
* Model (Conv→ReLU→Pool→Dense) ve **Flatten vs GAP** (neden GAP çoğu zaman daha iyi)
* Derleme (loss/optimizer/metric), Callbacks (EarlyStopping/Checkpoint)
* Eğitim (`fit`) → öğrenme eğrileri
* Test (accuracy/AUC), **Confusion Matrix**, ROC–PR
* Eşik (threshold) ayarı ve yanlışlara bakma
* Donanım: RAM/CPU/GPU nasıl kullanılıyor, neleri etkiler?

En altta **tam çalışan kod** var; yorumlar satır içi. (Senin deneyiminde: **Val AUC ≈ 0.959, Test AUC ≈ 0.956, Acc ≈ 0.892** — çok iyi bir temel.)

---

# 1) CNN nedir? (hızlı ama sağlam zihin modeli)

* **Conv2D**: Küçük filtre (örn. 3×3) görüntü üstünde kayar; **ağırlık paylaşımı** sayesinde parametre az, **konumdan bağımsız** özellik yakalanır (kenar/doku/parça).
* **ReLU**: Doğrusal olmayanlık; negatifleri 0’a kırpar → hızlı ve stabil öğrenme.
* **Pooling (MaxPool2D)**: Uzamsal boyutu yarıya indirir (2×2); küçük kaymalara dayanıklılık + parametre/hesap maliyeti azalır.
* **Flatten / GAP**: Konvolüsyonun ürettiği özellik haritalarını sınıflandırıcıya verir.

  * **Flatten** özelliği tek vektöre açar (çok parametre doğurabilir).
  * **GAP (GlobalAveragePooling)** her kanalın ortalamasını alır → **parametre çok azalır**, genelleme güçlenir.
* **Çıkış**: İkili sınıflama için `Dense(1, sigmoid)` + `binary_crossentropy`.

---

# 2) Veri hazırlığı — neden böyle?

**Klasör düzeni** (Keras’ın otomatik etiketlemesi için):

```
data/
  train/
    cats/
    dogs/
  val/
    cats/
    dogs/
  test/
    cats/
    dogs/
```

* **Neden Train/Val/Test?**

  * *Train*: öğrenme
  * *Val*: hiperparametre/erken durdurma için **dürüst** ölçüm
  * *Test*: final skor (hiç dokunma)

* **PetImages’ta bozuk dosyalar var.** `PIL.Image.verify()` ile **kopyalarken** bozukları ayıklıyoruz.

* **Denge**: Cats/Dogs sayıları zaten benzer; class\_weight gerekmez.

---

# 3) Data generator — ne yapıyor, neden önemli?

* **`ImageDataGenerator`** diskten **batch batch** okur → RAM’e tüm veri **yüklenmez** (sadece o anki batch).
* **Augmentation (sadece train)**: Döndürme, kaydırma, zoom, flip… “Aynı kedi/köpek farklı pozlarda da **aynı sınıf**” bilgisini öğretir → **overfitting azalır**, **genelleme artar**.
* **Rescale**: `[0,255] → [0,1]` ölçek → eğitim stabil.

**Boyut neden 160×160?**

* CNN sabit boyut ister. 128–160 **hız/performans** için çok iyi denge. (224 daha iyi olabilir ama yavaş ve bellek pahalı.)

**Augmentation doz mantığı (Cats vs Dogs için güvenli aralık):**

* `rotation_range=10–20`, `width/height_shift=0.05–0.15`, `zoom_range=0.10–0.20`, `shear_range=0.05–0.15`, `horizontal_flip=True`
* Fazlası **etiket gürültüsü** üretir (performans düşer).

---

# 4) Model — taş taş ve kritik kararlar

* **`padding='same'`**: Conv sonrası **genişlik/yükseklik korunur** (takip kolay, kenar bilgisi kalır).
* **Filtre sayısı artar (32→64→128)**: Aşağı indikçe daha **soyut** özellikler → temsil gücü.
* **3×3 kernel**: Ucuz ve etkili; üst üste kondukça “gördüğü alan” büyür.
* **Flatten mı GAP mı?**

  * Flatten + Dense(128) bu topolojiyle **\~6.5M parametre** (yavaş + overfit riski).
  * **GAP** ile **\~60×** daha az parametre; pratikte **daha hızlı** ve **genelde daha iyi** genelleme.

> Senin eğitiminde Flatten kullandın ve gayet iyi sonuç aldın. Bir sonraki denemede GAP’a geçersen benzer/iyileşmiş sonuçları **daha hızlı** görürsün.

---

# 5) Derleme & Callbacks

* **Loss**: `binary_crossentropy` (sigmoid ile doğal eş)
* **Optimizer**: `adam` (hızlı, güvenli başlangıç)
* **Metric**: `accuracy` + **`AUC`** (eşikten bağımsız ayrıştırma gücü)
* **EarlyStopping** (`monitor='val_auc'`, `patience=5`, `restore_best_weights=True`)
* **ModelCheckpoint** (en iyi `val_auc`’ta **kaydet**)

---

# 6) Eğitim (`fit`) & Öğrenme Eğrileri

* `fit(train_gen, validation_data=val_gen, epochs=30, callbacks=[...])`
* Train/Val **loss** birlikte düşüyorsa süper. Train çok düşüp Val bozulursa → **overfit**: augment ↑, **Dropout ↑**, GAP’a geç, LR ↓.

> Senin log: **Val AUC ≈ 0.959**, **Val Acc \~0.89** — güzel, dengeli yakınsama.

---

# 7) Test, CM, ROC–PR, yanlışlara bakma

* Test skorlarını yaz, **Confusion Matrix** ile hangi sınıf karışıyor gör.
* ROC–PR eğrileri ile ayrıştırmayı/precision–recall dengesini değerlendir.
* Yanlışları **görsel olarak** incele (en “özgüvenli” yanlışları gridde göster); augment/temizlik kararına ışık tutar.

> Senin test: **Loss 0.269, Acc 0.892, AUC 0.956**, CM dengeli (cats FN=195, dogs FN=212) — **gayet iyi**.

---

# 8) Eşik (threshold) ayarı

* Varsayılan 0.5 → dengeli.
* Maksimum accuracy / F1 için küçük bir tarama ile eşiği **ince ayarla** (örn. 0.52 → kedileri “köpek” deme oranı azalabilir).

---

# 9) Donanım — RAM/CPU/GPU nasıl kullanılıyor?

* **Disk → CPU → GPU akışı**:

  1. Generator dosyayı diskten okur, **CPU** ile resize & augment & `/255`.
  2. Batch numpy (float32) olarak **GPU’ya** (varsa) kopyalanır.
  3. İleri/geri yayılım **GPU’da** hesaplanır (CUDA/cuDNN).
* **RAM kullanımı**: Tüm dataset **RAM’e yüklenmez**; sadece o anki batch + ara aktivasyonlar hafızadadır.

  * Giriş batch bellek ≈ `batch × H × W × C × 4 byte` → `32 × 160 × 160 × 3 × 4 ≈ 9.8 MB` (aktivasyonlar hariç).
* **CPU hızlandırma**: `workers=4, use_multiprocessing=True` → okuma/augment paralel.
* **GPU büyük fark yaratır**. GPU yoksa `IMG_SIZE↓`, `BATCH_SIZE↓` ile sürede iyileşme.
* **Mixed Precision** (uyumlu NVIDIA GPU): eğitim hızlanır, bellek düşer:

  ```python
  from tensorflow.keras import mixed_precision
  mixed_precision.set_global_policy('mixed_float16')
  ```

  (Çıkış katmanı float32 kalır; Keras otomatik halleder.)

---

# Tam çalışan kod (PetImages → Train/Val/Test → CNN → Eğitim → Test/CM/ROC–PR → Eşik)

> **Not:** Bu sürüm **GAP** kullanır (önerilen). İstersen `use_gap=False` yapıp Flatten’lı sürümü de tek satırda deneyebilirsin. Bozuk görselleri kopyalarken eler.

```python
# =========================
# 0) Kurulum ve kütüphaneler
# =========================
import os, random, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator, image
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.metrics import AUC

from sklearn.metrics import (confusion_matrix, classification_report, roc_auc_score,
                             roc_curve, precision_recall_curve, average_precision_score)

np.random.seed(42); tf.random.set_seed(42); random.seed(42)
print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

# =========================
# 1) Kaynak klasörleri -> data/train|val|test  (%70|15|15) + bozuk dosyaları ele
# =========================
SOURCE = Path("PetImages")          # Kaggle zipten çıkan
CAT_DIR = SOURCE/"Cat"
DOG_DIR = SOURCE/"Dog"

TARGET = Path("data")
for split in ["train","val","test"]:
    for cls in ["cats","dogs"]:
        (TARGET/split/cls).mkdir(parents=True, exist_ok=True)

def list_valid_images(folder: Path):
    paths = []
    for p in folder.iterdir():
        if p.is_file():
            try:
                with Image.open(p) as im:
                    im.verify()  # içerik doğrulama (okuma değil)
                paths.append(p)
            except Exception:
                # bozuk / trunc / hatalı uzantı -> atla
                pass
    return paths

def split_copy(files, cls_name):
    random.shuffle(files)
    n = len(files)
    n_train = int(0.70*n)
    n_val   = int(0.15*n)
    parts = {
        "train": files[:n_train],
        "val"  : files[n_train:n_train+n_val],
        "test" : files[n_train+n_val:]
    }
    for split, items in parts.items():
        for src in items:
            dst = TARGET/split/cls_name/src.name
            if not dst.exists():
                shutil.copy2(src, dst)

# Yalnız ilk kurulumda aç:
if len(list((TARGET/"train"/"cats").glob("*"))) == 0:
    cats = list_valid_images(CAT_DIR)
    dogs = list_valid_images(DOG_DIR)
    print(f"Geçerli kedi: {len(cats)} | köpek: {len(dogs)}")
    split_copy(cats, "cats")
    split_copy(dogs, "dogs")
    for split in ["train","val","test"]:
        n_c = len(list((TARGET/split/"cats").glob("*")))
        n_d = len(list((TARGET/split/"dogs").glob("*")))
        print(split, "-> cats:", n_c, "dogs:", n_d)

# =========================
# 2) Data generators (augment sadece train)
# =========================
IMG_SIZE   = (160,160)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    shear_range=0.10,
    zoom_range=0.10,
    horizontal_flip=True,
)

val_datagen  = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TARGET/"train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    seed=42
)
val_gen = val_datagen.flow_from_directory(
    TARGET/"val",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)
test_gen = test_datagen.flow_from_directory(
    TARGET/"test",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print("Sınıf indeksleri:", train_gen.class_indices)  # {'cats':0, 'dogs':1}

# =========================
# 3) Model (GAP önerilir) — Flatten istersen use_gap=False yap
# =========================
def make_cnn(input_shape=(160,160,3), dropout=0.3, use_gap=True):
    m = models.Sequential()
    m.add(layers.Conv2D(32, 3, activation='relu', padding='same', input_shape=input_shape))
    m.add(layers.MaxPool2D(2))

    m.add(layers.Conv2D(64, 3, activation='relu', padding='same'))
    m.add(layers.MaxPool2D(2))

    m.add(layers.Conv2D(128, 3, activation='relu', padding='same'))
    m.add(layers.MaxPool2D(2))
    
    if use_gap:
        # Parametre dostu, genelleme iyi
        m.add(layers.GlobalAveragePooling2D())
        m.add(layers.Dropout(dropout))
        m.add(layers.Dense(128, activation='relu', kernel_initializer='he_normal'))
    else:
        # Klasik Flatten (parametre sayısı çok artar)
        m.add(layers.Flatten())
        m.add(layers.Dropout(0.5))
        m.add(layers.Dense(128, activation='relu'))

    m.add(layers.Dense(1, activation='sigmoid'))

    m.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy', AUC(name='auc')])
    return m

model = make_cnn(input_shape=IMG_SIZE+(3,), dropout=0.3, use_gap=True)
model.summary()

# =========================
# 4) Callbacks ve Eğitim (fit)
# =========================
early = EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True, verbose=1)
ckpt  = ModelCheckpoint("best_cnn.keras", monitor='val_auc', mode='max', save_best_only=True, verbose=1)
plateau = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)

EPOCHS = 30
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=[early, ckpt, plateau],
    verbose=1,
    # workers=4, use_multiprocessing=True,  # CPU I/O hızlandırma
)

# =========================
# 5) Öğrenme eğrileri
# =========================
hist = history.history
plt.figure(figsize=(11,4))
plt.subplot(1,2,1); plt.plot(hist['loss']); plt.plot(hist['val_loss']); plt.title('Loss'); plt.legend(['train','val']); plt.grid(alpha=0.3)
plt.subplot(1,2,2); plt.plot(hist['accuracy']); plt.plot(hist['val_accuracy']); plt.title('Accuracy'); plt.legend(['train','val']); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# =========================
# 6) Test, CM, ROC–PR
# =========================
best_model = tf.keras.models.load_model("best_cnn.keras")

test_loss, test_acc, test_auc = best_model.evaluate(test_gen, verbose=0)
print(f"TEST -> Loss: {test_loss:.4f} | Acc: {test_acc:.4f} | AUC: {test_auc:.4f}")

probs = best_model.predict(test_gen).ravel()
y_pred = (probs >= 0.5).astype(int)
y_true = test_gen.classes
class_names = list(test_gen.class_indices.keys())

print("\nClassification report:\n", classification_report(y_true, y_pred, target_names=class_names))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

# ROC & PR
fpr, tpr, _ = roc_curve(y_true, probs)
ap = average_precision_score(y_true, probs)

plt.figure(figsize=(11,4))
plt.subplot(1,2,1)
plt.plot(fpr, tpr, label=f"AUC={roc_auc_score(y_true, probs):.3f}")
plt.plot([0,1],[0,1],'k--'); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC"); plt.legend(); plt.grid(alpha=0.3)

prec, rec, _ = precision_recall_curve(y_true, probs)
plt.subplot(1,2,2)
plt.plot(rec, prec, label=f"AP={ap:.3f}")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision-Recall"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# =========================
# 7) Eşik (threshold) optimizasyonu (maks acc)
# =========================
ts = np.linspace(0.3, 0.7, 41)
best_acc, best_t = 0, 0.5
for t in ts:
    yp = (probs >= t).astype(int)
    acc = (yp==y_true).mean()
    if acc > best_acc: best_acc, best_t = acc, t
print(f"Best accuracy @ threshold={best_t:.2f}: {best_acc:.4f}")

# =========================
# 8) Yanlışları görsel inceleme (en "özgüvenli" yanlışlar)
# =========================
filenames = test_gen.filenames
wrong_idx = np.where(y_pred != y_true)[0]
conf = np.abs(probs[wrong_idx] - 0.5)
top = wrong_idx[np.argsort(-conf)[:12]]

plt.figure(figsize=(10,9))
for i, idx in enumerate(top, 1):
    img_path = test_gen.directory + "/" + filenames[idx]
    img = image.load_img(img_path, target_size=test_gen.target_size)
    plt.subplot(3,4,i); plt.imshow(img); plt.axis("off")
    plt.title(f"pred={y_pred[idx]} (p1={probs[idx]:.2f})\ntrue={y_true[idx]}")
plt.tight_layout(); plt.show()

# =========================
# 9) Tek görsel tahmini (aynı ön işleme!)
# =========================
idx_to_cls = {v:k for k,v in train_gen.class_indices.items()}

def predict_one(img_path, img_size=IMG_SIZE):
    img = image.load_img(img_path, target_size=img_size)
    arr = image.img_to_array(img)/255.0
    arr = np.expand_dims(arr, axis=0)
    p = float(best_model.predict(arr)[0][0])
    pred = idx_to_cls[1] if p >= 0.5 else idx_to_cls[0]
    print(f"{img_path} -> {pred} (prob '{idx_to_cls[1]}'={p:.3f})")

# predict_one("PetImages/Cat/1.jpg")
# predict_one("PetImages/Dog/1.jpg")
```

---

## Kısa “sonuç yorumu” şablonu (kendi çıktına uygula)

* **Val AUC \~ 0.95+**, **Test AUC \~ 0.95+** ⇒ **genelleme güçlü**
* **Test Acc \~ 0.89** ⇒ Cats/Dogs görevinde **sağlam baseline**
* **CM** dengeli ⇒ Sınıflar arasında yanılgılar simetrik
* **Yanlışların motifleri**: Karanlık/uzak/blur/alışılmadık pozlar → augment’i ışık/kontrastla azıcık zenginleştir, bozukları temizle
* **Hız/kalite takası**: `IMG_SIZE 160`, `GAP` → hızlı/sağlam; istersen 192–224 deneyip batch’i düşür
* **Daha ileri**: MobileNetV2/EfficientNetB0 ile **transfer learning** (freeze→fine-tune) ≈ %93–95+ acc çok olası

---

bundan sonra gönül rahatlığıyla bir sonraki eğitime geçebilirsin. İstersen bu projeyi **transfer learning** ile 10–15 dakikalık bir ek deneme olarak tamamlayıp, iki modelin metriklerini **tek tabloda** kıyaslamayı da gösterebilirim.
